# Day 66: Feature Engineering — Encoding Categorical Variables

## Objective
- Learn why ML models need numbers instead of text
- Know when to use Label, Ordinal, and One-Hot encoding
- Avoid the dummy variable trap

## Imports

In [1]:
# Standard data tools
import pandas as pd

# Encoders from scikit-learn
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder

## Theory
- **Nominal** data has no order (position, league, foot)
- **Ordinal** data has a real order (low < medium < high)
- Models only understand numbers, so text columns must be encoded
- Wrong encoding = the model learns a **fake relationship**
- **Dummy variable trap:** one-hot columns that add up to 1 are perfectly predictable from each other (perfect multicollinearity), so we drop one with `drop_first=True`

In [2]:
# Load the small football dataset
df = pd.read_csv("dataset.csv")
print(df.head())
print(df.dtypes)

   player_id    position preferred_foot fitness_level      league  goals  \
0          1     Forward          Right          high     Premier     18   
1          2  Midfielder           Left        medium     La Liga      6   
2          3    Defender          Right           low     Serie A      1   
3          4  Goalkeeper          Right        medium  Bundesliga      0   
4          5     Forward           Left          high     La Liga     22   

   market_value_m  
0            42.5  
1            25.0  
2            12.0  
3            18.5  
4            55.0  
player_id           int64
position              str
preferred_foot        str
fitness_level         str
league                str
goals               int64
market_value_m    float64
dtype: object


## Example 1: LabelEncoder (and why it is risky for nominal data)

In [3]:
# LabelEncoder turns each category into a number
le = LabelEncoder()
df["position_label"] = le.fit_transform(df["position"])

# See which number each position got
print({name: int(num) for name, num in zip(le.classes_, le.transform(le.classes_))})
# Problem: the model may think Midfielder (3) > Defender (0)

{'Defender': 0, 'Forward': 1, 'Goalkeeper': 2, 'Midfielder': 3}


In [4]:
# LabelEncoder on an ORDINAL column: it sorts A-Z, not by meaning!
le2 = LabelEncoder()
df["fitness_label"] = le2.fit_transform(df["fitness_level"])
print({name: int(num) for name, num in zip(le2.classes_, le2.transform(le2.classes_))})
# high=0, low=1, medium=2 -> wrong order!

{'high': 0, 'low': 1, 'medium': 2}


## Example 2: OrdinalEncoder with a custom order

In [5]:
# We tell sklearn the real order: low -> medium -> high
oe = OrdinalEncoder(categories=[["low", "medium", "high"]])
df["fitness_ordinal"] = oe.fit_transform(df[["fitness_level"]])
print(df[["fitness_level", "fitness_ordinal"]].drop_duplicates().sort_values("fitness_ordinal"))

  fitness_level  fitness_ordinal
2           low              0.0
1        medium              1.0
0          high              2.0


## Example 3: One-Hot Encoding

In [6]:
# pandas way: one binary column per category
dummies = pd.get_dummies(df["position"], dtype=int)
print(dummies.head(3))

   Defender  Forward  Goalkeeper  Midfielder
0         0        1           0           0
1         0        0           0           1
2         1        0           0           0


In [7]:
# sklearn way: same idea, but it remembers the categories (fit / transform)
ohe = OneHotEncoder(sparse_output=False)
arr = ohe.fit_transform(df[["league"]])
print(ohe.get_feature_names_out())
print(arr[:3])

['league_Bundesliga' 'league_La Liga' 'league_Premier' 'league_Serie A']
[[0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]]


## Example 4: The Dummy Variable Trap

In [8]:
# With all columns kept, each row always sums to 1 (redundant info)
full = pd.get_dummies(df["preferred_foot"], dtype=int)
print("Row sums:", full.sum(axis=1).unique())

# drop_first=True removes one column -> no perfect multicollinearity
safe = pd.get_dummies(df["preferred_foot"], drop_first=True, dtype=int)
print(safe.columns.tolist())

Row sums: [1]
['Right']


## Practice Exercise
Use `pd.get_dummies()` on the `league` column with `drop_first=True`. How many new columns do you get, and why is it one fewer than the number of leagues?

In [9]:
# Your turn: one-hot encode 'league' safely
league_dummies = pd.get_dummies(df["league"], drop_first=True, dtype=int)
print(league_dummies.shape[1], "columns")
print(league_dummies.columns.tolist())

3 columns
['La Liga', 'Premier', 'Serie A']


## Mini Challenge
Build one clean feature table: encode `fitness_level` with OrdinalEncoder, one-hot encode `position` and `league` with `drop_first=True`, and keep `goals`.

In [10]:
# Combine every encoding into one model-ready table
ready = pd.concat([
    df[["goals"]],
    df[["fitness_ordinal"]],
    pd.get_dummies(df[["position", "league"]], drop_first=True, dtype=int)
], axis=1)
print(ready.shape)
print(ready.head(3))

(20, 8)
   goals  fitness_ordinal  position_Forward  position_Goalkeeper  \
0     18              2.0                 1                    0   
1      6              1.0                 0                    0   
2      1              0.0                 0                    0   

   position_Midfielder  league_La Liga  league_Premier  league_Serie A  
0                    0               0               1               0  
1                    1               1               0               0  
2                    0               0               0               1  


## Summary
- Models need numbers, so categorical text must be encoded
- **Ordinal data** -> OrdinalEncoder with a custom order
- **Nominal data** -> One-Hot Encoding (not LabelEncoder)
- LabelEncoder sorts alphabetically, which can break ordinal meaning
- Use `drop_first=True` to avoid the dummy variable trap